**Tokenizer**

- byte pair encoding (BPE)
  - encode the sequence
  - get the most likely pairs of bytes, then merge them (e.g. (101,104) -> 257)
  - add that pair to a list called 'merges' and add it to your vocab dict
  - do that num_merges times
- when you want to encode something
  - encode it normally in utf-8
  - then walk the 'merges' list using a for loop - and begin merging tokens on every pass until you've done all the 'merges' in the 'merges' list
  - its important you walk the list in a for loop, because some merges may be of (257, 102) for example, which is a merging of an already merged token (i.e. 257)

_Here's an intuitive diagram!_

- each step is a pass of the text data, where we merge the first pair in our 'merges' list, then the second...
- we do this $\text{num merges}$ times, since that's how many merges we did during training to construct our new vocab_size!

_PS: typical vocab_size is 50,257, including padding tokens + < bos > tokens_

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQUQ8_fwiCtMQr-aVf-Ii1mhRPCccnz6627SLUhUmwtYJFMSWmAAoALVz0Q&s=10" width="400">


**What to consider when tokenizing**

Vocab size vs encoding efficiency

- you can have a few ids (i.e. 256 for example using default Unicode 8 encoding), which is one byte for every character
- BUT that means you'll need a lot of token_ids to represent one string of text (i.e. 10,000 character extract -> 10,000 token_ids)

The trade is to have more token_ids, and more encoding efficiency

- now we could expand our vocabulary (i.e. from 256 -> 50,257 word encodings) meaning we'd go from one byte per character to 1-2 bytes representing 3-4+ characters (depending on the relative frequency of characters and pairs in our training data)
- now it might take only 4,000 token_ids to encode a 10,000 character extract

_Why this is good?_

- computationally cheaper - your sequence length is smaller so the next token you predict is a bigger chunk of text
- much easier for the model to learn relationships - consider trying to embed the semantic meaning of '-able' vs '-b', like what even is the semantic meaning of the letter 'b'???

_Why is this bad_

- our LM head and our embedding matrix are ENORMOUS! That's why early GPT-2 used weight-tying (i.e. make them the exact same weights)


In [ ]:
data = """
“If you’re going to try, go all the way.

Otherwise, don’t even start.

This could mean losing girlfriends, wives, relatives and maybe even your mind.

It could mean not eating for three or four days.

It could mean freezing on a park bench.

It could mean jail.

It could mean derision.

It could mean mockery — isolation.

Isolation is a gift.

All the others are a test of your endurance, of how much you really want to do it.

And, you’ll do it, despite rejection and the worst odds.

And it will be better than anything else you can imagine.

If you’re going to try, go all the way.

There is no other feeling like that.

You will be alone with the gods, and the nights will flame with fire.

You will ride life straight to perfect laughter.

It’s the only good fight there is.”

- Charles Bukowski (1920 – 1994) 🪦
"""  

In [ ]:
from collections import defaultdict


class BPETokenizer:

	def __init__(self, vocab_size:int):

		self.merge_mapping = {}
		self.vocab_size = vocab_size
		assert vocab_size >= 256, f"BPE needs vocab size greater than 256"

		self.vocab = {i: bytes([i]) for i in range(256)} # mapping from byte to index - initially its just 1:1, 2:2, but after merging you will get 259: (223, 24)
		self.merges = defaultdict(int)

	def get_stats(self, ids):

		# then of those tokens, now merge the ones that have mergeed pairs
		counts = defaultdict(int)
		for pair in zip(ids, ids[1:]):
			counts[pair] += 1

		return counts

	def merge_seq(self, ids, pair, idx):
		newids = []
		i = 0
		while i < len(ids):
			if i < len(ids) - 1 and ids[i]==pair[0] and ids[i+1]==pair[1]:
				i += 2
				newids.append(idx)
			else:
				newids.append(ids[i]) # append the current token
				i += 1 # move to the next
		return newids
	
	def merge(self, ids, num_merges: int):
		
		for i in range(num_merges):
			stats = self.get_stats(ids)
			freq_pair = max(self.counts, key=self.counts.get) # most frequent pair
			idx = 256 + i
			print(f"merged tokens {freq_pair} to token id: {256+i}")
			ids = self.merge_seq(ids, freq_pair, 256+i)
			self.merges[freq_pair] = idx
		
		for (p0, p1), idx in self.merges.items():
			
			# concat bytes objects 
			self.vocab[idx] = self.vocab[p0] + self.vocab[p1]
		
	
	def decode(self, ids):
		
		# decode a sequence 
		tokens = b"".join(self.vocab[idx] for idx in ids)
		text = tokens.decode("utf-8", errors="replace") 
		return text
		 

	def _encode(self, text):
		
		ids = list(text.encode('utf-8'))


		while len(ids) >= 2:
			stats = self.get_stats(ids)
			pair = min(stats, key=lambda p: self.merges.get(p, float('-inf'))) # using the keys of 'stats', get their value in the 'merges' array, and find the minimum value - i.e. lowest idx
			
			if pair in self.merges: 
				self.merges.get(pair)
				ids = self.merge_seq(ids, pair, idx)
			else: 
				break

	def encode(self, text):

		ids = list(text.encode('utf-8'))

		for pair, idx in self.merges.items(): # (259, 3455), 5000 -> combined_tokens, token_idx
			for i in range(len(ids)): # iterate through all tokens 
				ids = self.merge_seq(ids, pair, idx)

		return ids

bpe = BPETokenizer(vocab_size=256)
hello = bpe.encode(data)
bpe.decode(hello)